# 노드 A — tonight

**6 GPU = 3노드 x 2 GPU. 이 노트북은 노드 A 전용.**

세 노드 전부 `exp5_tonight.py` 하나만 부른다. 잡 정의 · 우선순위 · 스킵 · 집계가 전부 거기 있다.
**세 노트북은 내용이 동일하고 노드 문자만 다르다** — 학습으로 배정되든 eval로 배정되든 이 하나로 된다.

**예산**: eval 1셀(LIBERO-10 x 50ep/task = 500ep) ≈ 2 GPU-h · 학습 1잡(150k) ≈ 8 GPU-h

---

## 오늘의 목표 — stride sweep (8/25 해준님이 은지님께 약속한 것)

8/25 K=100 결과(은지님, n=500, 100k):

| stride | ACT | ACM2 | BiMamba | carry | BiMOS |
|---|---|---|---|---|---|
| 10 | 42.5 | 56.8 | **65.0** | 60.0 | 60.4 |
| 50 | 23.6 | 37.4 | 31.2 | **36.8** | 31.2 |
| 100 | 18.4 | 18.6 | 21.6 | **27.8** | 19.6 |

**모든 stride에서 ACT를 이기고, s10에서 +22.5.** 같은 stride = 같은 추론 호출 횟수라
동일 비용 우위다. 그런데 **봉우리 양옆(s5·s15·s25)이 비어 있어** 이게 진짜 봉우리인지,
ACT가 s5에서 역전하지는 않는지 모른다. 그걸 채우는 게 오늘 1순위.

같이 확보된 것(전부 K=100 s10, 이 세팅을 고정하는 근거):
- **스캔 ablation**: plain 56.8 / scan_same(2배 파라미터) 56.0 / BiMamba 65.0
  → 성능은 파라미터가 아니라 **역방향 스캔** 덕 (+8.2). 교수님 요청 실험, n=500
- **backbone**: r18 65.0 vs r50 51.0 (−14.0) → ResNet18 유지
- **decoder layer**: L1 59.1 / L2 63.4 / L4 48.6 → 2층

## 우선순위

1. **K=100 stride sweep** — s5 → s15 → s25 → (s10/50/100은 완료분 skip) → s75 → s1
   × 6변형(act / acm2 / bimamba순수 / bimamba_cpoff / bimos / carry)
2. **K sweep @ stride=10 고정** — 메인 그림. 지금 이 축에 K=50(ACT 67.6 / BiMamba 57.8)과
   K=100(42.5 / 65.0) 두 점뿐이라 크로스오버가 50~100 사이라는 것만 알고 곡선이 없다.
   ACT ckpt는 전 K에 있어 **재학습 없이 eval만**으로 채워진다.
3. K=50 stride sweep (67.6 재현 + carry 표 모순 해소)
4. TE 축 500ep 확정 런 (act+TE는 8/25에 100ep로 측정: 전 K에서 우리가 위)

## 학습 (critical — eval 큐 깊이와 무관하게 1노드 선점)

- **`acm2` @ K=100** — stride sweep의 ACM2 열 전체를 막고 있다. 스캔 ablation의 분모
- **`bimamba_pure_k10/15`** — 현재 bimamba 값은 carry-학습 오염판(cpoff)


## 1) 부팅

In [ ]:
import sys
from pathlib import Path

_h = Path.cwd()
_r = next(c for c in (_h, *_h.parents) if (c / 'notebooks' / 'libero' / 'exp5_tonight.py').exists())
for _p in (_r / 'notebooks', _r / 'notebooks' / 'libero'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import exp5_tonight as X
X = importlib.reload(X)
cf, v23 = X.setup()          # common_final reload + 태그 등록 (순서 중요)

NODE = 'A'
# 이 노드에서 쓸 GPU. 한 노드에서 창을 2개 띄울 때만 [2, 3] 처럼 직접 지정.
GPUS = v23.available_gpus([0, 1])
print('NODE', NODE, '| GPUS', GPUS)


## 2) 인벤토리 — 뭐가 학습돼 있고 뭐가 없나

서버 파일시스템을 실제로 스캔한다. `MISS` = 학습 필요, `PART` = 중단됨(resume 대상).
`MISS`/`PART`인 태그를 참조하는 eval 셀은 자동으로 큐에서 빠진다.

In [ ]:
rows = X.inventory()


## 3) 역할 배정 ← **여기가 오늘 뭘 할지 정한다**

규칙: ① critical 학습이 노드를 **1대까지만** 선점 → ② 나머지는 ready eval 큐 포화 → ③ 그래도 남으면 일반 학습.
ready eval이 24셀 이상이면 선점은 1노드로 묶인다 (stride sweep을 굶기지 않도록).
예상 배정: **A = 학습(acm2, bimamba_pure), B·C = eval(stride sweep).**

출력이 "기본값과 다르다"면 알려주는 `X.ROLE_OVERRIDE = ...` 한 줄을 **세 노트북 모두**에 넣어야 잡이 안 겹친다.

In [ ]:
roles = X.suggest()

# 위 출력이 "기본값과 다르다" 라고 하면, 알려주는 한 줄을 **세 노트북 모두**에 붙여넣고
# 이 셀부터 다시 실행할 것. (세 노드가 같은 역할표를 봐야 잡이 안 겹친다)
# X.ROLE_OVERRIDE = {'C': 'train'}


## 4) 계획 — 이 노드 몫

실행 전에 목록을 눈으로 확인할 것.

In [ ]:
plan = X.plan(NODE, GPUS)


## 5) preflight (약 12분) — **셋 중 한 노드에서 한 번만**

override(`n_action_steps`/`temporal_ensemble_coeff`)가 실제로 먹는지 확인.
exp3가 같은 방식(TE override)으로 이미 정상 완주했으므로 사실상 통과 확인용이다.
다른 노드에서 이미 통과했으면 건너뛸 것.

In [ ]:
X.preflight(gpu=GPUS[0], n_ep=5)


## 6) eval — 3)에서 `eval`로 배정됐을 때

먼저 dry-run으로 커맨드를 보고 실행. `GPUS` 수만큼 청크로 돌고 청크마다 블로킹한다.
로그는 `outputs/final/_logs/exp5__*.log`. **아침에 다시 실행하면 완료분은 skip되고 이어서 돈다.**

In [ ]:
X.run_evals(plan['eval'][:2], plan['gpus'], dry=True)


In [ ]:
if X.role_of(NODE) != 'eval':
    print('노드 ' + NODE + ' 는 train 으로 배정됐다 -> 7)번 학습 셀을 쓸 것. 여기는 건너뛴다.')
else:
    X.run_evals(plan['eval'], plan['gpus'])


## 7) 학습 — 3)에서 `train`으로 배정됐을 때

**dry-run에서 반드시 확인**: `bimamba_pure_*` 커맨드에 `--use_chunk_pairs`가 **없고** `--policy.sscp_enabled=false`가 **있어야** 한다. 이거 하나 틀리면 8시간을 날린다.

실행 셀은 잡이 끝날 때까지(~8h/잡) 블로킹한다.

In [ ]:
X.run_trains(plan['train'], plan['gpus'], dry=True)


In [ ]:
if X.role_of(NODE) != 'train':
    print('노드 ' + NODE + ' 는 eval 로 배정됐다 -> 6)번 eval 셀을 쓸 것. 여기는 건너뛴다.')
elif not plan['train']:
    print('학습 큐가 비었다.')
else:
    X.run_trains(plan['train'], plan['gpus'])


## 8) 이어서 — 다음 배치

위 배치가 끝나면 인벤토리가 바뀐다. 이 셀로 역할·계획을 다시 뽑고 6) 또는 7)로 돌아간다.

In [ ]:
# 위 배치가 끝나면 인벤토리가 바뀐다. 이 셀로 역할/계획을 다시 뽑고 6) 또는 7)로 돌아간다.
_ov = dict(X.ROLE_OVERRIDE)      # reload 하면 초기화되므로 수동 오버라이드를 보존한다
X = importlib.reload(X)
cf, v23 = X.setup(verbose=False)
X.ROLE_OVERRIDE = _ov
roles = X.suggest()
plan = X.plan(NODE, GPUS)


## 아침에 볼 것

`X.report()` 가 **stride sweep → K sweep → TE 축** 순으로 찍는다.

| 결과 | 다음 |
|---|---|
| s5/s15에서도 BiMamba > ACT | **봉우리가 감싸졌다** → K=100 s10을 메인 세팅으로 확정 |
| ACT가 s5에서 역전 | 봉우리가 왼쪽 → s1~s5를 더 촘촘히 |
| K sweep에서 ACT가 K≥100부터 무너짐 | 교수님이 요청한 "ACT가 무너지는 지점" — 메인 그림 확정 |
| 순수 `bimamba` ≈ `bimamba_cpoff` | 오염 무해 → 은지님 기존 값(75.4 등) 그대로 인용 가능 |
| `bimos` < `bimamba` | carry는 메인에서 빼고 ablation으로 (K=100 s10에서 이미 −4.6) |

**주의**: 50ep/task(= overall 500ep) 기준 SE ≈ ±2.2%p. 5%p 미만 차이는 seed/rep을 늘린 뒤에만 주장할 것.

In [ ]:
X.report()
